# QUELL Step 08 — Gorulmemis-attack genellemesi (leave-one-attack-out)

**Q1 kancasi deneyi.** Her attack classini sirayla trainingden cikarir, test'te 'hic unseen' haliyle gosterir. RF ve LLM ayni sekilde kapali-kume egitilir, tahmin ikiliye (benign/attack) indirgenir; unseen attack 'attack' diye classlanirsa yakalanmis sayilir (**unseen recall**).

Hipotez: LLM'in on-trainingli anlamsal temsili yeni attacklara RF'ten daha iyi genellesin. Her fold sonrasi JSON'a kaydeder (cokerse kaldigi yerden devam). Tek hucreyi calistir; sonda OZET tablosunu paylas.

Ayarlar hucrenin basinda: hizli deneme icin `MAX_FOLDS=3` yap; tam kosu icin `None` birak.

In [ ]:
# ================== QUELL Step 08 — UNKNOWN-ATTACK GENERALIZATION (leave-one-attack-out) ==================
# Idea: her attack classini sirayla EGITIMDEN doneen cikar, test'te "hic unseen" haliyle goster.
# Both RF and LLM are trained closed-set; the prediction is reduced to binary (benign / attack).
# Gorulmemis attack "attack" diye classlanirsa YAKALANMIS sayilir (unseen recall).
# Q1 hypothesis: does the LLM's semantic representation generalize to new attack patterns better than RF?
import os, subprocess
try:
    _o=subprocess.check_output("nvidia-smi --query-gpu=index,memory.free --format=csv,noheader,nounits",shell=True,text=True)
    _f=[(int(x.split(",")[0]),int(x.split(",")[1])) for x in _o.strip().splitlines()]
    _b=max(_f,key=lambda t:t[1]); os.environ["CUDA_VISIBLE_DEVICES"]=str(_b[0])
    os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
    print("selected GPU:",_b[0],"| free(MiB):",_f,flush=True)
except Exception as e: print("GPU secim atlandi:",e)
import json, time, sys, gc
from pathlib import Path
import numpy as np, pandas as pd
from pandas.api.types import is_numeric_dtype
for pk in ["transformers","peft","accelerate","bitsandbytes"]:
    try: __import__(pk)
    except Exception: subprocess.run([sys.executable,"-m","pip","install","-q",pk])
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# ===== AYARLAR =====
DATASET="nbaiot"; MODEL="Qwen/Qwen2.5-1.5B"
TRAIN_CAP=1500; EPOCHS=1; SEED=42          # 1 epoch/1500 to keep each fold fast (set 2/2000 if you want)
BENIGN_FPR_CAP=5000; KNOWN_CAP=5000        # cap on benign/known-attack samples in evaluation
MAX_FOLDS=None                              # None = ALL attack classes; for a quick trial set 3
# ===================
torch.manual_seed(SEED); np.random.seed(SEED)
ROOT=Path.home()/"quell-edge-llm-ids"; PROC=ROOT/"data"/"processed"; SPL=ROOT/"splits"; RES=ROOT/"results"
rep=json.load(open(RES/"split_report.json")); meta=rep[DATASET]
label=meta["label_col"]; group=meta.get("group_col"); tcol=meta.get("time_col")
LABELISH={"label","attack","attack_type","attack_label","type","class","category","marker","__label__"}
df=pd.read_parquet(PROC/f"{DATASET}.parquet").reset_index(drop=True)
sp=np.load(SPL/f"{DATASET}_split.npz"); tr_idx,te_idx=sp["train"],sp["test"]
drop=set([label])|set(meta.get("leaky_candidates",[]))
if group: drop.add(group)
if tcol: drop.add(tcol)
for c in df.columns:
    if c!=label and c.lower() in LABELISH: drop.add(c)
feats=[c for c in df.columns if c not in drop]
y_all=df[label].astype(str).values
classes_all=sorted(pd.unique(y_all).tolist())

# --- find the benign (harmless) class ---
benign=None
for c in classes_all:
    lc=c.strip().lower()
    if lc in {"normal","benign","background"} or "normal" in lc or "benign" in lc:
        benign=c; break
assert benign is not None, f"benign class not found. classes: {classes_all}"
attacks=[c for c in classes_all if c!=benign]
if MAX_FOLDS: attacks=attacks[:MAX_FOLDS]
print(f"{DATASET}: benign='{benign}' | {len(attacks)} attack folds | feature={len(feats)}",flush=True)

# --- numeric matrix for RF (factorize object columns) ---
Xdf=df[feats].copy()
for c in Xdf.columns:
    if not is_numeric_dtype(Xdf[c]): Xdf[c]=pd.factorize(Xdf[c])[0]
X=np.nan_to_num(Xdf.values.astype("float64"),nan=0.0,posinf=0.0,neginf=0.0)
X=np.clip(X,-1e9,1e9).astype("float32")  # N-BaIoT asiri buyukleri kirp (tasma/inf onle)

# --- feature->text for the LLM ---
def row_to_text(r):
    parts=[]
    for c in feats:
        v=r[c]
        if isinstance(v,(float,np.floating)): v=round(float(v),4)
        parts.append(f"{c}={v}")
    return "Network traffic flow. "+", ".join(parts)+" . Attack type:"
print("feature->text...",flush=True)
texts_all=df.apply(row_to_text,axis=1).values
MAX_LEN=256 if len(feats)<=64 else 512
BF16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()

def balanced_train(tr_pool):
    rng=np.random.default_rng(SEED); sel=[]
    for cls in pd.unique(y_all[tr_pool]):
        ids=tr_pool[y_all[tr_pool]==cls]
        if len(ids)>TRAIN_CAP: ids=rng.choice(ids,TRAIN_CAP,replace=False)
        sel+=ids.tolist()
    return np.array(sorted(sel))

def eval_rows_for(H):
    r=np.random.default_rng(SEED)
    H_rows=te_idx[y_all[te_idx]==H]
    ben=te_idx[y_all[te_idx]==benign]
    if len(ben)>BENIGN_FPR_CAP: ben=r.choice(ben,BENIGN_FPR_CAP,replace=False)
    oth=te_idx[(y_all[te_idx]!=H)&(y_all[te_idx]!=benign)]
    if len(oth)>KNOWN_CAP: oth=r.choice(oth,KNOWN_CAP,replace=False)
    return H_rows,ben,oth

def train_llm(tr_sel,le):
    K=len(le.classes_)
    tok=AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    class DS(torch.utils.data.Dataset):
        def __init__(s,idx): s.idx=idx
        def __len__(s): return len(s.idx)
        def __getitem__(s,i):
            j=s.idx[i]; e=tok(texts_all[j],truncation=True,max_length=MAX_LEN,padding="max_length",return_tensors="pt")
            it={k:v.squeeze(0) for k,v in e.items()}; it["labels"]=torch.tensor(int(le.transform([y_all[j]])[0])); return it
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
    base=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=K,quantization_config=bnb,device_map={"":0})
    base=prepare_model_for_kbit_training(base); base.config.pad_token_id=tok.pad_token_id
    model=get_peft_model(base,LoraConfig(task_type=TaskType.SEQ_CLS,r=16,lora_alpha=32,lora_dropout=0.05,
        target_modules=["q_proj","v_proj"],modules_to_save=["score"]))
    model.config.use_cache=False
    args=TrainingArguments(output_dir=str(ROOT/"models"/"tmp_gen"),per_device_train_batch_size=8,
        gradient_accumulation_steps=2,num_train_epochs=EPOCHS,learning_rate=2e-4,bf16=BF16,fp16=(not BF16),
        gradient_checkpointing=True,logging_steps=100,save_strategy="no",report_to=[],seed=SEED)
    Trainer(model=model,args=args,train_dataset=DS(tr_sel)).train()
    return model,tok

def llm_predict(model,tok,idx):
    dev=next(model.parameters()).device; model.eval(); preds=[]; bs=64
    for s in range(0,len(idx),bs):
        js=idx[s:s+bs]
        enc=tok(list(texts_all[js]),truncation=True,max_length=MAX_LEN,padding=True,return_tensors="pt").to(dev)
        with torch.no_grad(), torch.autocast(device_type="cuda",dtype=torch.bfloat16,enabled=(dev.type=="cuda")):
            lo=model(**enc).logits
        preds+=lo.argmax(-1).cpu().tolist()
    return np.array(preds)

def rates(pred_labels,H_rows,ben,oth):
    # pred_labels: string tahminler; benign disi => "attack" (yakalandi)
    def det(mask_labels): return float(np.mean(mask_labels!=benign)) if len(mask_labels) else float("nan")
    return dict(unseen_recall=round(det(pred_labels["H"]),4),
                benign_fpr=round(det(pred_labels["ben"]),4),
                known_recall=round(det(pred_labels["oth"]),4))

OUT=RES/"generalization_report.json"
allg=json.load(open(OUT)) if OUT.exists() else {}
allg.setdefault(DATASET,{"benign":benign,"model":MODEL,"config":{"train_cap":TRAIN_CAP,"epochs":EPOCHS},"folds":{}})
folds=allg[DATASET]["folds"]

for fi,H in enumerate(attacks,1):
    if H in folds:
        print(f"[{fi}/{len(attacks)}] {H} already exists, skipping",flush=True); continue
    t0=time.time()
    tr_pool=tr_idx[y_all[tr_idx]!=H]           # remove H entirely from training
    tr_sel=balanced_train(tr_pool)
    le=LabelEncoder().fit(y_all[tr_sel])        # no H -> the model never sees H
    H_rows,ben,oth=eval_rows_for(H)
    n_H=len(H_rows)
    print(f"\n[{fi}/{len(attacks)}] UNSEEN='{H}'  train={len(tr_sel):,}  test H={n_H:,} benign={len(ben):,} known={len(oth):,}",flush=True)

    # --- RF ---
    rf=RandomForestClassifier(n_estimators=200,n_jobs=-1,class_weight="balanced",random_state=SEED)
    rf.fit(X[tr_sel],y_all[tr_sel])
    rf_pred={"H":rf.predict(X[H_rows]),"ben":rf.predict(X[ben]),"oth":rf.predict(X[oth])}
    rf_r=rates(rf_pred,H_rows,ben,oth)
    print(f"    RF : unseen_recall={rf_r['unseen_recall']}  benign_fpr={rf_r['benign_fpr']}  known_recall={rf_r['known_recall']}",flush=True)

    # --- LLM ---
    model,tok=train_llm(tr_sel,le)
    lp=lambda idx: le.inverse_transform(llm_predict(model,tok,idx))
    llm_pred={"H":lp(H_rows),"ben":lp(ben),"oth":lp(oth)}
    llm_r=rates(llm_pred,H_rows,ben,oth)
    print(f"    LLM: unseen_recall={llm_r['unseen_recall']}  benign_fpr={llm_r['benign_fpr']}  known_recall={llm_r['known_recall']}",flush=True)
    del model,tok; gc.collect(); torch.cuda.empty_cache()

    folds[H]={"n_unseen":int(n_H),"rf":rf_r,"llm":llm_r,"sec":round(time.time()-t0,0)}
    json.dump(allg,open(OUT,"w"),indent=2,ensure_ascii=False)   # save after each fold (crash protection)
    print(f"    saved ({time.time()-t0:.0f}s) -> results/generalization_report.json",flush=True)

# --- OZET ---
print("\n================ UNKNOWN-ATTACK SUMMARY (unseen recall) ================")
print(f"{'attack':28s} {'n':>7s} {'RF':>7s} {'LLM':>7s}  {'kazanan':>8s}")
rf_u=[]; llm_u=[]
for H,v in folds.items():
    a,b=v["rf"]["unseen_recall"],v["llm"]["unseen_recall"]; rf_u.append(a); llm_u.append(b)
    win="LLM" if b>a else ("RF" if a>b else "=")
    print(f"{H[:28]:28s} {v['n_unseen']:>7d} {a:>7.3f} {b:>7.3f}  {win:>8s}")
if rf_u:
    print("-"*64)
    print(f"{'ORTALAMA':28s} {'':>7s} {np.mean(rf_u):>7.3f} {np.mean(llm_u):>7.3f}")
    print(f"\nLLM ortalama unseen recall - RF = {np.mean(llm_u)-np.mean(rf_u):+.3f}  (pozitif = LLM daha iyi geneller)")
print("DONE.")
